# 태스크 2: Strands를 사용하여 다중 에이전트 워크플로 구축

이 작업에서는 Strands Agents의 [’Agents as Tools’](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/multi-agent/agents-as-tools/) 패턴을 사용하여 단일 에이전트 시스템에서 정교한 다중 에이전트 아키텍처로 발전하게 됩니다. 오케스트레이터 아래 협력하여 포괄적인 재무 지침을 제공하는 전문 금융 에이전트를 구성하게 됩니다.

이 태스크는 전문 에이전트가 협업하여 단일 상담원의 능력을 넘어서는 복잡한 문제를 해결하는 엔터프라이즈급 에이전트 조정을 보여줍니다. 프로덕션 규모의 AI 시스템 구축에 필수적인 기술인 계층적 에이전트 설계, 도구 래핑 패턴, 지능형 요청 라우팅을 배우게 됩니다.

![아키텍처](./images/multi-agent_ko_kr.png)


### 시스템 아키텍처

멀티 에이전트 시스템은 세 가지 핵심 구성 요소로 구성됩니다.

#### 1. 예산 에이전트(태스크 1)
**개인 예산 책정, 지출 분석 및 재무 규율을 전문으로 합니다.**

| 도구 | 설명 | 예제 사용 사례 |
| --- | --- | --- |
| **calculate_budget_breakdown** | 모든 소득 수준에 대한 50/30/20 예산 계산 | 월 소득 6,000달러에 대한 예산 생성 |
| **analyze_spending_pattern** | 맞춤형 추천을 통한 지출 패턴 분석 | 소득 5,000달러 대비 800달러의 식사 비용 분석. |
| **calculator** | 재무 계산 및 수학 연산 | 예산의 20% 절감 목표를 계산 |
  
#### 2. 재무 분석 에이전트
**투자 조사, 포트폴리오 관리 및 시장 분석에 중점을 둡니다.**

| 도구 | 설명 | 예제 사용 사례 |
|------|-------------|------------------|
| **get_stock_analysis** | 실시간 주식 데이터 및 종합 분석 | Apple 주식 실적 및 지표를 분석해 주세요. |
| **create_diversified_portfolio** | 배분을 포함한 위험 기반 포트폴리오 권장 사항 | 10,000달러 예산의 중간 위험 포트폴리오 생성 |
| **compare_stock_performance** | 기간별 다중 주식 실적 비교 | 6개월 동안 Tesla, Apple, Google 비교 |

#### 3. 오케스트레이터 에이전트

**전문 에이전트를 조정하고 포괄적인 응답을 종합합니다.**

| 능력 | 설명 | 예제 사용 사례 |
|------------|-------------|------------------|
| **에이전트 라우팅** | 상담할 전문가를 지능적으로 결정 | 예산 관련 질문은 예산 담당자에게, 투자 문의는 재무 담당자에게 라우팅 |
| **다중 에이전트 조정** | 복잡한 쿼리에 대해 여러 에이전트의 통찰력을 결합 | 두 에이전트를 함께 사용하는 ‘예산 책정 및 투자 지원’ |
| **응답 종합** | 여러 에이전트의 결과로부터 일관된 응답 생성 | 예산 분석과 투자 권장 사항 결합 |
| **컨텍스트 관리** | 에이전트 상호작용 전반에 걸쳐 대화 흐름 유지 | 후속 권장 사항 작성 시 이전 조언 기억 |

In [ ]:
%%capture
# Install required dependencies for multi-agent system
!pip install --force-reinstall -U -r requirements.txt --quiet --disable-pip-version-check

In [ ]:
# Import dependencies for multi-agent system and stock analysis
from strands import Agent, tool
from strands.models import BedrockModel
from strands.agent.conversation_manager import SummarizingConversationManager
import yfinance as yf
from typing import List
from utils import create_guardrail
import boto3
import logging

# Get the current AWS region dynamically
region = boto3.Session().region_name

# Configure logging for error tracking and debugging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

### 태스크 2.1: 예산 에이전트를 도구로 활용

In [ ]:
# Wrap budget agent from Task 1 as a tool for orchestration
from budget_agent import FinancialReport, budget_agent

@tool
def budget_agent_tool(query: str) -> FinancialReport:
    """Generate structured financial reports with budget analysis and recommendations."""
    try:
        structured_response = budget_agent.structured_output(
            output_model=FinancialReport, prompt=query
        )
        return structured_response
    except Exception as e:
        # Return a default structured response on error
        return FinancialReport(
            monthly_income=0.0,
            budget_categories=[],
            recommendations=[f"Error generating report: {str(e)}"],
            financial_health_score=1,
        )

### 태스크 2.2: 재무 분석 에이전트 생성

In [ ]:
# Define system prompt for investment-focused financial analysis agent
FINANCIAL_ANALYSIS_PROMPT = """귀하는 투자 리서치 및 포트폴리오 추천을 전문으로 하는 금융 분석 에이전트입니다. 귀하의 역할은 다음과 같습니다.

1. 주식 성과 데이터 조사 및 분석
2. 분산 투자 포트폴리오 구성
3. 데이터 기반의 투자 제안 제공

귀하는 구체적인 투자 조언을 직접 제공하는 대신, 사용자가 합리적인 결정을 내릴 수 있도록 분석 데이터를 제시합니다. 항상 시장 리스크와 금융 전문가와의 상담 필요성에 대한 면책 ​​조항을 포함해야 합니다."""

In [ ]:
# Configure Bedrock model for financial analysis agent
bedrock_model = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    region_name=region,
    temperature=0.0,  # Deterministic responses for financial advice
)

In [ ]:
# Tool for comprehensive stock analysis using yfinance
@tool
def get_stock_analysis(symbol: str) -> str:
    """Get comprehensive analysis for a specific stock symbol."""
    try:
        stock = yf.Ticker(symbol)
        info = stock.info
        hist = stock.history(period="1y")

        # Calculate key metrics
        current_price = hist["Close"].iloc[-1]
        year_high = hist["High"].max()
        year_low = hist["Low"].min()
        avg_volume = hist["Volume"].mean()
        price_change = (
            (current_price - hist["Close"].iloc[0]) / hist["Close"].iloc[0]
        ) * 100

        return f"""
📊 Stock Analysis for {symbol.upper()}:
• Current Price: ${current_price:.2f}
• 52-Week High: ${year_high:.2f}
• 52-Week Low: ${year_low:.2f}
• Year-to-Date Change: {price_change:.2f}%
• Average Daily Volume: {avg_volume:,.0f} shares
• Company: {info.get("longName", "N/A")}
• Sector: {info.get("sector", "N/A")}
"""
    except Exception as e:
        return f"❌ Unable to retrieve data for {symbol}: {str(e)}"

In [ ]:
# Tool for creating risk-based diversified investment portfolios
@tool
def create_diversified_portfolio(risk_level: str, investment_amount: float) -> str:
    """Create a diversified portfolio based on risk level (conservative, moderate, aggressive) and investment amount."""

    portfolios = {
        "conservative": {
            "stocks": ["JNJ", "PG", "KO", "PEP", "WMT"],
            "weights": [0.25, 0.20, 0.20, 0.20, 0.15],
            "description": "Stable, dividend-paying blue-chip stocks with low volatility",
        },
        "moderate": {
            "stocks": ["AAPL", "MSFT", "JPM", "V", "DIS"],
            "weights": [0.25, 0.25, 0.20, 0.15, 0.15],
            "description": "Mix of established tech leaders and stable financial/consumer stocks",
        },
        "aggressive": {
            "stocks": ["TSLA", "NVDA", "META", "COIN", "PLTR"],
            "weights": [0.30, 0.25, 0.20, 0.15, 0.10],
            "description": "High-growth tech and emerging sector stocks with higher volatility",
        },
    }

    if risk_level.lower() not in portfolios:
        return "❌ Risk level must be: conservative, moderate, or aggressive"

    portfolio = portfolios[risk_level.lower()]

    result = f"""
🎯 {risk_level.upper()} Portfolio Recommendation (${investment_amount:,.0f}):
{portfolio["description"]}

Portfolio Allocation:
"""

    for stock, weight in zip(portfolio["stocks"], portfolio["weights"]):
        allocation = investment_amount * weight
        result += f"• {stock}: {weight * 100:.0f}% (${allocation:,.0f})\n"

    result += "\n⚠️ Disclaimer: This is for educational purposes only. Consult a financial advisor before investing."
    return result

In [ ]:
# Tool for comparing performance of multiple stocks over time periods
@tool
def compare_stock_performance(symbols: List[str], period: str = "1y") -> str:
    """Compare performance of multiple stocks over a specified period (1y, 6m, 3m, 1m)."""
    if len(symbols) > 5:
        return "❌ Please limit comparison to 5 stocks maximum"

    try:
        performance_data = {}

        for symbol in symbols:
            stock = yf.Ticker(symbol)
            hist = stock.history(period=period)
            if not hist.empty:
                start_price = hist["Close"].iloc[0]
                end_price = hist["Close"].iloc[-1]
                performance = ((end_price - start_price) / start_price) * 100
                performance_data[symbol] = performance

        result = f"📈 Stock Performance Comparison ({period}):\n"
        sorted_stocks = sorted(
            performance_data.items(), key=lambda x: x[1], reverse=True
        )

        for stock, performance in sorted_stocks:
            result += f"• {stock}: {performance:+.2f}%\n"

        return result

    except Exception as e:
        return f"❌ Error comparing stocks: {str(e)}"

In [ ]:
# Create financial analysis agent with investment tools
financial_analysis_agent = Agent(
    model=bedrock_model,  # Using the same bedrock_model from Step 1
    system_prompt=FINANCIAL_ANALYSIS_PROMPT,
    tools=[get_stock_analysis, create_diversified_portfolio, compare_stock_performance],
)

In [ ]:
# Test financial analysis agent with portfolio and stock analysis
response = financial_analysis_agent(
    "10,000달러 규모의 중위험 포트폴리오를 구성하고 애플 주식을 분석하세요."
)

In [ ]:
%%writefile financial_analysis_agent.py
# Export financial analysis agent to standalone Python file

import yfinance as yf
from strands import Agent, tool
from typing import List
from strands.models import BedrockModel
import logging
import boto3

# Get the current AWS region dynamically
region = boto3.Session().region_name

# Configure logging for error tracking and debugging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Financial Analysis Agent System Prompt
FINANCIAL_ANALYSIS_PROMPT = """귀하는 투자 리서치 및 포트폴리오 추천을 전문으로 하는 금융 분석 에이전트입니다. 귀하의 역할은 다음과 같습니다.

1. 주식 성과 데이터 조사 및 분석
2. 분산 투자 포트폴리오 구성
3. 데이터 기반의 투자 제안 제공

귀하는 구체적인 투자 조언을 직접 제공하는 대신, 사용자가 합리적인 결정을 내릴 수 있도록 분석 데이터를 제시합니다. 항상 시장 리스크와 금융 전문가와의 상담 필요성에 대한 면책 ​​조항을 포함해야 합니다."""

bedrock_model = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    region_name=region,
    temperature=0.0,  # Deterministic responses for financial advice
)


# Tool 1: Get Stock Analysis
@tool
def get_stock_analysis(symbol: str) -> str:
    """Get comprehensive analysis for a specific stock symbol."""
    try:
        # Fetch stock data
        stock = yf.Ticker(symbol)
        info = stock.info
        hist = stock.history(period="1y")
        
        if hist.empty:
            return f"❌ Error: No data found for symbol '{symbol.upper()}'."
        
        # Calculate key metrics
        current_price = hist["Close"].iloc[-1]
        year_high = hist["High"].max()
        year_low = hist["Low"].min()
        avg_volume = hist["Volume"].mean()
        price_change = ((current_price - hist["Close"].iloc[0]) / hist["Close"].iloc[0]) * 100

        return f"""
📊 Stock Analysis for {symbol.upper()}:
• Current Price: ${current_price:.2f}
• 52-Week High: ${year_high:.2f}
• 52-Week Low: ${year_low:.2f}
• Year-to-Date Change: {price_change:.2f}%
• Average Daily Volume: {avg_volume:,.0f} shares
• Company: {info.get("longName", "N/A")}
• Sector: {info.get("sector", "N/A")}
"""
    except Exception as e:
        logger.error(f"Error retrieving data for {symbol}: {e}")
        return f"❌ Error: Unable to retrieve stock data for '{symbol}'."


# Tool 2: Create Diversified Portfolio with comprehensive error handling
@tool
def create_diversified_portfolio(risk_level: str, investment_amount: float) -> str:
    """위험 수준(보수형, 중립형, 공격형)과 투자 금액을 바탕으로 분산 투자 포트폴리오를 구성하십시오."""
    try:

        # Convert to float for calculations
        amount = float(investment_amount)
        
        # Input validation - negative check
        if amount < 0:
            logger.warning(f"Negative investment amount: {amount}")
            return "❌ Error: Investment amount cannot be negative. Please provide a positive amount."
        
        # Input validation - zero check
        if amount == 0:
            return "❌ Error: Investment amount cannot be zero. Please provide a positive amount to invest."
        
        # Input validation - minimum investment
        if amount < 100:
            return "❌ Error: Investment amount too small. Please provide at least $100 for portfolio diversification."
        
        # Input validation - maximum investment
        if amount > 100_000_000:
            logger.warning(f"Unusually high investment amount: {amount}")
            return "❌ Error: Investment amount seems unusually high. Please verify the amount (maximum $100,000,000)."
        
        portfolios = {
            "conservative": {
                "stocks": ["JNJ", "PG", "KO", "PEP", "WMT"],
                "weights": [0.25, 0.20, 0.20, 0.20, 0.15],
                "description": "Stable, dividend-paying blue-chip stocks with low volatility",
            },
            "moderate": {
                "stocks": ["AAPL", "MSFT", "JPM", "V", "DIS"],
                "weights": [0.25, 0.25, 0.20, 0.15, 0.15],
                "description": "Mix of established tech leaders and stable financial/consumer stocks",
            },
            "aggressive": {
                "stocks": ["TSLA", "NVDA", "META", "COIN", "PLTR"],
                "weights": [0.30, 0.25, 0.20, 0.15, 0.10],
                "description": "High-growth tech and emerging sector stocks with higher volatility",
            },
        }
        
        # Input validation - risk level check
        risk_level_lower = risk_level.strip().lower()
        if risk_level_lower not in portfolios:
            logger.warning(f"Invalid risk level provided: {risk_level}")
            return "❌ Error: Risk level must be 'conservative', 'moderate', or 'aggressive'. Please choose one of these options."
        
        portfolio = portfolios[risk_level_lower]
        
        result = f"""
🎯 {risk_level_lower.upper()} Portfolio Recommendation (${amount:,.0f}):
{portfolio["description"]}

Portfolio Allocation:
"""
        
        for stock, weight in zip(portfolio["stocks"], portfolio["weights"]):
            allocation = amount * weight
            result += f"• {stock}: {weight * 100:.0f}% (${allocation:,.0f})\n"
        
        result += "\n⚠️ Disclaimer: This is for educational purposes only. Consult a financial advisor before investing."
        return result
    
    except (ValueError, TypeError) as e:
        logger.error(f"Invalid input for create_diversified_portfolio: risk_level={risk_level}, amount={investment_amount}, error: {e}")
        return "❌ Error: Unable to create portfolio. Please check your inputs and try again."
    except Exception as e:
        logger.error(f"Unexpected error in create_diversified_portfolio: {e}")
        return "❌ Error: An unexpected error occurred while creating portfolio. Please try again."


# Tool 3: Compare Stock Performance with comprehensive error handling
@tool
def compare_stock_performance(symbols: List[str], period: str = "1y") -> str:
    """Compare performance of multiple stocks over a specified period (1y, 6m, 3m, 1m)."""
    try:
        # Input validation - type checking
        if not isinstance(symbols, list):
            logger.error(f"Invalid type for symbols: {type(symbols)}")
            return "❌ Error: Symbols must be provided as a list"
        
        # Input validation - empty list check
        if not symbols:
            return "❌ Error: Please provide at least one stock symbol to compare."
        
        # Input validation - list size check
        if len(symbols) > 5:
            return "❌ Error: Please limit comparison to 5 stocks maximum for better readability."
        
        # Input validation - period check
        valid_periods = ["1m", "3m", "6m", "1y", "2y", "5y"]
        if period not in valid_periods:
            logger.warning(f"Invalid period provided: {period}")
            return f"❌ Error: Invalid time period '{period}'. Please use one of: {', '.join(valid_periods)}"
        
        # Validate each symbol
        for symbol in symbols:
            if not isinstance(symbol, str):
                logger.error(f"Non-string symbol in list: {symbol}")
                return "❌ Error: All stock symbols must be text strings"
            if not symbol.strip():
                return "❌ Error: Empty stock symbol found. Please provide valid ticker symbols."
        
        performance_data = {}
        failed_symbols = []
        
        for symbol in symbols:
            symbol = symbol.strip().upper()
            try:
                stock = yf.Ticker(symbol)
                hist = stock.history(period=period)
                
                if not hist.empty and len(hist) >= 2:
                    start_price = hist["Close"].iloc[0]
                    end_price = hist["Close"].iloc[-1]
                    performance = ((end_price - start_price) / start_price) * 100
                    performance_data[symbol] = performance
                else:
                    failed_symbols.append(symbol)
            except Exception as e:
                logger.warning(f"Failed to fetch data for {symbol}: {e}")
                failed_symbols.append(symbol)
        
        # Check if we got any valid data
        if not performance_data:
            return f"❌ Error: Unable to retrieve data for any of the provided symbols. Please verify the ticker symbols are correct."
        
        result = f"📈 Stock Performance Comparison ({period}):\n"
        sorted_stocks = sorted(
            performance_data.items(), key=lambda x: x[1], reverse=True
        )
        
        for stock, performance in sorted_stocks:
            result += f"• {stock}: {performance:+.2f}%\n"
        
        # Add note about failed symbols if any
        if failed_symbols:
            result += f"\n⚠️ Note: Unable to retrieve data for: {', '.join(failed_symbols)}"
        
        return result
    
    except ValueError as e:
        logger.error(f"Value error in compare_stock_performance: {e}")
        return "❌ Error: Invalid input values. Please check your stock symbols and period."
    except Exception as e:
        logger.error(f"Unexpected error in compare_stock_performance: {e}")
        return "❌ Error: Unable to compare stock performance. Please check your internet connection and try again."


# Create the Financial Analysis Agent
financial_analysis_agent = Agent(
    model=bedrock_model,  # Using the same bedrock_model from Step 1
    system_prompt=FINANCIAL_ANALYSIS_PROMPT,
    tools=[get_stock_analysis, create_diversified_portfolio, compare_stock_performance],
    callback_handler=None,
)

if __name__ == "__main__":
    # Test the Financial Analysis Agent
    response = financial_analysis_agent(
        "10,000달러 규모의 중위험 포트폴리오를 구성하고 애플 주식을 분석하세요."
    )
    print(response)

In [ ]:
# Test standalone financial analysis agent
!python financial_analysis_agent.py 

### 태스크 2.3: 재무 분석 에이전트를 도구로 활용

In [ ]:
# Wrap financial analysis agent as tool for orchestrator
from financial_analysis_agent import financial_analysis_agent


@tool
def financial_analysis_agent_tool(query: str) -> str:
    """Handle investment analysis queries including stock research, portfolio creation, and performance comparisons."""
    try:
        response = financial_analysis_agent(query)
        return str(response)
    except Exception as e:
        return f"❌ Financial analysis error: {str(e)}"

### 태스크 2.4: 오케스트레이터 에이전트 생성

In [ ]:
# Define orchestrator system prompt for multi-agent coordination
ORCHESTRATOR_PROMPT = """당신은 전문 금융 에이전트들을 조율하여 포괄적인 금융 조언을 제공하는 종합 금융 자문 코디네이터입니다.

당신이 활용하는 전문 에이전트는 다음과 같습니다:
1. **budget_agent**: 예산 수립, 지출 분석, 저축 제안 및 지출 내역 관리 담당
2. **financial_analysis_agent_tool**: 투자 분석, 주식 조사, 포트폴리오 구성 및 성과 비교 담당

에이전트 활용 지침:
- 예산, 지출 습관, 지출 내역 관리, 저축 목표, 부채 관리 관련 질문에는 **budget_agent**를 활용하세요.
- 주식, 투자, 포트폴리오, 시장 분석, 투자 제안 관련 질문에는 **financial_analysis_agent_tool**을 활용하세요.
- 종합적인 재무 설계를 위해 두 에이전트를 함께 활용할 수 있습니다.
- 여러 에이전트의 정보를 종합할 때는 항상 일관성 있는 요약 내용을 제공하세요.
- 도움이 되면서도 전문적인 어조를 유지하고, 금융 조언에 대한 적절한 면책 조항을 포함하세요.

사용자의 질문을 받았을 때:
1. 가장 적합한 에이전트를 결정합니다.
2. 구체적인 질의 내용을 바탕으로 해당 에이전트를 호출합니다.
3. 각 에이전트의 응답을 종합하여 일관성 있고 포괄적인 답변을 작성합니다.
4. 가능한 경우, 실행 가능한 후속 조치를 제안합니다."""

In [ ]:
# Create guardrail for orchestrator agent
guardrail_id, guardrail_arn = create_guardrail()

In [ ]:
# Configure Bedrock model for orchestrator with guardrails
bedrock_model = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    region_name=region,
    temperature=0.0,  # Deterministic responses for financial advice
    guardrail_id=guardrail_id,  # Your Bedrock guardrail ID
    guardrail_version="DRAFT",  # Guardrail version
    guardrail_trace="enabled",
)

In [ ]:
# Configure conversation manager for orchestrator
conversation_manager = SummarizingConversationManager(
    summary_ratio=0.3,  # Summarize 30% of messages when context reduction is needed
    preserve_recent_messages=5,  # Always keep 5 most recent messages
)

In [ ]:
# Create orchestrator agent with both specialized agent tools
orchestrator_agent = Agent(
    model=bedrock_model,
    system_prompt=ORCHESTRATOR_PROMPT,
    tools=[budget_agent_tool, financial_analysis_agent_tool],
    conversation_manager=conversation_manager,  # Associate a conversation manager
)

In [ ]:
# Test orchestrator with complex multi-agent query
response = orchestrator_agent("테슬라와 애플의 주식을 비교해 주시고, 월 소득 4,000달러로 2,000달러를 투자할 여력이 있는지 알려주세요.",)

In [ ]:
%%writefile main.py
# Export complete multi-agent system to main.py

from strands import Agent, tool
from strands.models import BedrockModel
from strands.agent.conversation_manager import SummarizingConversationManager

from budget_agent import FinancialReport, budget_agent
from financial_analysis_agent import financial_analysis_agent

from utils import get_guardrail_id
import boto3

# Get the current AWS region dynamically
region = boto3.Session().region_name

ORCHESTRATOR_PROMPT = """당신은 전문 금융 에이전트들을 조율하여 포괄적인 금융 조언을 제공하는 종합 금융 자문 코디네이터입니다.

당신이 활용하는 전문 에이전트는 다음과 같습니다:
1. **budget_agent**: 예산 수립, 지출 분석, 저축 제안 및 지출 내역 관리 담당
2. **financial_analysis_agent_tool**: 투자 분석, 주식 조사, 포트폴리오 구성 및 성과 비교 담당

에이전트 활용 지침:
- 예산, 지출 습관, 지출 내역 관리, 저축 목표, 부채 관리 관련 질문에는 **budget_agent**를 활용하세요.
- 주식, 투자, 포트폴리오, 시장 분석, 투자 제안 관련 질문에는 **financial_analysis_agent_tool**을 활용하세요.
- 종합적인 재무 설계를 위해 두 에이전트를 함께 활용할 수 있습니다.
- 여러 에이전트의 정보를 종합할 때는 항상 일관성 있는 요약 내용을 제공하세요.
- 도움이 되면서도 전문적인 어조를 유지하고, 금융 조언에 대한 적절한 면책 조항을 포함하세요.

사용자의 질문을 받았을 때:
1. 가장 적합한 에이전트를 결정합니다.
2. 구체적인 질의 내용을 바탕으로 해당 에이전트를 호출합니다.
3. 각 에이전트의 응답을 종합하여 일관성 있고 포괄적인 답변을 작성합니다.
4. 가능한 경우, 실행 가능한 후속 조치를 제안합니다."""

# Add conversation management to maintain context
conversation_manager = SummarizingConversationManager(
    summary_ratio=0.3,  # Summarize 30% of messages when context reduction is needed
    preserve_recent_messages=5,  # Always keep 5 most recent messages
)

# Continue with previous configurations
bedrock_model = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    region_name=region,
    temperature=0.0,  # Deterministic responses for financial advice
    guardrail_id=get_guardrail_id(),
    guardrail_version="DRAFT",
    guardrail_trace="enabled",
)


@tool
def budget_agent_tool(query: str) -> FinancialReport:
    """Generate structured financial reports with budget analysis and recommendations."""
    try:
        structured_response = budget_agent.structured_output(
            output_model=FinancialReport, prompt=query
        )
        return structured_response
    except Exception as e:
        # Return a default structured response on error
        return FinancialReport(
            monthly_income=0.0,
            budget_categories=[],
            recommendations=[f"Error generating report: {str(e)}"],
            financial_health_score=1,
        )


# Wrap Financial Analysis Agent as a Tool
@tool
def financial_analysis_agent_tool(query: str) -> str:
    """Handle investment analysis queries including stock research, portfolio creation, and performance comparisons."""
    try:
        response = financial_analysis_agent(query)
        return str(response)
    except Exception as e:
        return f"❌ Financial analysis error: {str(e)}"


orchestrator_agent = Agent(
    model=bedrock_model,
    system_prompt=ORCHESTRATOR_PROMPT,
    tools=[budget_agent_tool, financial_analysis_agent_tool],
    conversation_manager=conversation_manager,
)

if __name__ == "__main__":
    orchestrator_agent = Agent(
        model=bedrock_model,
        system_prompt=ORCHESTRATOR_PROMPT,
        tools=[budget_agent_tool, financial_analysis_agent_tool],
    )

    response = orchestrator_agent("월 6,000달러를 벌고 있으며, 매달 500달러씩 투자를 시작하고 싶습니다. 예산 계획을 세우고 투자 포트폴리오를 추천해 주세요.")

In [ ]:
# Test complete multi-agent orchestrator system
!python main.py 

**태스크 완료:** 예산 책정 및 투자 분석을 위해 전문 에이전트를 조정하는 계층적 다중 에이전트 시스템을 성공적으로 구축했습니다. 이제 시스템은 지능형 라우팅 및 조정을 통해 여러 전문 분야가 필요한 복잡한 쿼리를 처리할 수 있습니다.

### 다음 단계

이 노트북을 완료했습니다. 실습의 다음 부분으로 넘어가려면 실습 지침으로 돌아가서 **태스크 3**을 계속하십시오.